# Cats vs dogs with json-camera

**Read this before running anything.**

This notebook does not sell you the library. It runs the experiment you would
run, on 300 cats and 300 dogs, and reports what actually happened. The headline
result is that **for a dataset this size you should not use compressed-domain
training at all**, and the notebook shows you why with numbers.

Measured on an M1 Pro, 600 images, identical architecture where comparable:

| approach | test accuracy | time |
|---|---|---|
| pixels, trained from scratch | 51.7% | 289s |
| **latents**, trained from scratch | 54.2% | 187s + 20s to convert |
| **pretrained ResNet18, frozen** | **94.2%** | **88s** |

Both from-scratch runs are at chance (50% for two classes). Six hundred images
is nowhere near enough to learn cats from dogs from a cold start, so the 9x
speedup is 9x faster at learning nothing. Transfer learning wins by 40 points
and is quicker, and **the latent path cannot use it**, because a pretrained
backbone expects 3 channels at 224x224 and a latent is 128 channels at 14x14.

So when *is* it worth it? Section 5 works out the break-even. Short version:
training from scratch, tens of thousands of images upward, where disk or I/O
hurts and no pretrained backbone fits your input.


## 1. Setup

Any folder in `ImageFolder` layout works:

```
catsdogs/train/cats/*.jpg    catsdogs/train/dogs/*.jpg
catsdogs/test/cats/*.jpg     catsdogs/test/dogs/*.jpg
```

The cell below builds one from Oxford-IIIT Pet if you have no data of your own.


In [1]:
import os, glob, random, shutil, time
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.set_num_threads(4)

ROOT = 'data/catsdogs'      # point this at your own folder if you have one
PER_CLASS_TRAIN, PER_CLASS_TEST = 240, 60


In [2]:
# Build the split from Oxford-IIIT Pet. Skip if ROOT already exists.
# In that dataset a capitalised filename is a cat breed, lowercase is a dog.
if not os.path.isdir(ROOT):
    src = glob.glob('data/pets/images/*.jpg')
    assert src, 'download Oxford-IIIT Pet into data/pets/images first'
    cats = sorted(p for p in src if os.path.basename(p)[0].isupper())
    dogs = sorted(p for p in src if not os.path.basename(p)[0].isupper())
    random.shuffle(cats); random.shuffle(dogs)
    n = PER_CLASS_TRAIN + PER_CLASS_TEST
    for split, sl in [('train', slice(0, PER_CLASS_TRAIN)), ('test', slice(PER_CLASS_TRAIN, n))]:
        for name, paths in [('cats', cats), ('dogs', dogs)]:
            d = f'{ROOT}/{split}/{name}'; os.makedirs(d, exist_ok=True)
            for p in paths[sl]: shutil.copy(p, d)

for s in ('train', 'test'):
    print(s, {c: len(os.listdir(f'{ROOT}/{s}/{c}')) for c in ('cats', 'dogs')})


train {'cats': 240, 'dogs': 240}
test {'cats': 60, 'dogs': 60}


## 2. The workflow you asked about

> *why turn them to json and redo or something*

You do not re-encode every epoch. **You convert once**, and the converted set is
what you train on from then on. Your original photos stay exactly where they are;
nothing is overwritten and nothing is thrown away that you cannot re-derive.

```
your photos  ──convert once──>  train.jcl  ──> epoch 1, 2, 3, ... 100
  (kept)         ~20 seconds      1.2 MB        each epoch reads the small file
```

The conversion is a fixed one-off cost. The saving is per epoch. That is the whole
economic argument, and section 5 shows where the two lines cross.


In [3]:
from jsoncam.dataset import prepare_dataset, LatentDataset

t0 = time.time()
for split in ('train', 'test'):
    prepare_dataset(f'{ROOT}/{split}', f'{split}.jcl', size=224, progress=False)
convert_seconds = time.time() - t0

pixels_mb  = sum(os.path.getsize(os.path.join(dp, f))
                 for s in ('train','test') for dp,_,fs in os.walk(f'{ROOT}/{s}') for f in fs) / 1e6
latents_mb = sum(os.path.getsize(f'{s}.jcl') for s in ('train','test')) / 1e6

ds = LatentDataset('train.jcl')
print(f'converted in {convert_seconds:.0f}s')
print(f'on disk: {pixels_mb:.1f} MB of photos -> {latents_mb:.1f} MB of latents  ({pixels_mb/latents_mb:.0f}x smaller)')
print(f'each sample is now a tensor of {ds.latent_shape}, not 3x224x224')


wrote train.jcl: 480 images, 1.2 MB (resized to 224px first, so not comparable to source files)


wrote test.jcl: 120 images, 0.3 MB (resized to 224px first, so not comparable to source files)
converted in 16s
on disk: 61.4 MB of photos -> 1.4 MB of latents  (42x smaller)
each sample is now a tensor of (128, 14, 14), not 3x224x224


## 3. Train both ways

Same network either way, apart from the input stem: the pixel version starts with
stride 2 on 3 channels, the latent version with stride 1 on 128. Everything after
that is identical, so the comparison is about the input and nothing else.


In [4]:
def make_net(in_ch, first_stride):
    return nn.Sequential(
        nn.Conv2d(in_ch, 64, 3, stride=first_stride, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(256, 2))

def train_and_score(net, train_dl, test_dl, epochs=12, lr=1e-3):
    opt = torch.optim.Adam(net.parameters(), lr); lossf = nn.CrossEntropyLoss()
    t0 = time.time()
    for _ in range(epochs):
        net.train()
        for x, y in train_dl:
            opt.zero_grad(); lossf(net(x), y).backward(); opt.step()
    seconds = time.time() - t0
    net.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_dl:
            correct += (net(x).argmax(1) == y).sum().item(); total += y.numel()
    return 100 * correct / total, seconds


In [5]:
# --- pixels ---
tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
px_train = DataLoader(datasets.ImageFolder(f'{ROOT}/train', tf), batch_size=32, shuffle=True, num_workers=2)
px_test  = DataLoader(datasets.ImageFolder(f'{ROOT}/test',  tf), batch_size=32, num_workers=2)
px_acc, px_s = train_and_score(make_net(3, 2), px_train, px_test)
print(f'pixels : {px_acc:.1f}%  in {px_s:.0f}s')


pixels : 57.5%  in 273s


In [6]:
# --- latents ---
C, H, W = LatentDataset('train.jcl').latent_shape
lt_train = DataLoader(LatentDataset('train.jcl'), batch_size=32, shuffle=True, num_workers=2)
lt_test  = DataLoader(LatentDataset('test.jcl'),  batch_size=32, num_workers=2)
lt_acc, lt_s = train_and_score(make_net(C, 1), lt_train, lt_test)
print(f'latents: {lt_acc:.1f}%  in {lt_s:.0f}s (+{convert_seconds:.0f}s conversion)')


latents: 52.5%  in 163s (+16s conversion)


## 4. The control nobody runs, and the one that wins

Before concluding anything about 9x, check what a normal person would actually do
with 600 images: take a network already trained on ImageNet, freeze it, and train
a two-class head on top.


In [7]:
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters(): p.requires_grad = False
m.fc = nn.Linear(512, 2)

norm = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
n_train = DataLoader(datasets.ImageFolder(f'{ROOT}/train', norm), batch_size=32, shuffle=True, num_workers=2)
n_test  = DataLoader(datasets.ImageFolder(f'{ROOT}/test',  norm), batch_size=32, num_workers=2)

opt = torch.optim.Adam(m.fc.parameters(), 1e-3); lossf = nn.CrossEntropyLoss()
t0 = time.time()
for _ in range(3):
    m.train()
    for x, y in n_train:
        opt.zero_grad(); lossf(m(x), y).backward(); opt.step()
tl_s = time.time() - t0
m.eval(); correct = total = 0
with torch.no_grad():
    for x, y in n_test:
        correct += (m(x).argmax(1) == y).sum().item(); total += y.numel()
tl_acc = 100 * correct / total
print(f'pretrained: {tl_acc:.1f}%  in {tl_s:.0f}s')


pretrained: 92.5%  in 87s


In [8]:
print(f"{'approach':32}{'accuracy':>10}{'time':>10}")
print(f"{'pixels, from scratch':32}{px_acc:9.1f}%{px_s:9.0f}s")
print(f"{'latents, from scratch':32}{lt_acc:9.1f}%{lt_s+convert_seconds:9.0f}s")
print(f"{'pretrained ResNet18, frozen':32}{tl_acc:9.1f}%{tl_s:9.0f}s")
print()
print('chance for two balanced classes is 50%.')


approach                          accuracy      time
pixels, from scratch                 57.5%      273s
latents, from scratch                52.5%      179s
pretrained ResNet18, frozen          92.5%       87s

chance for two balanced classes is 50%.


## 5. So when is any of this worth it?

**Not here.** Both from-scratch runs are at chance. The 9x speedup is real and it
is 9x faster at learning nothing, while a pretrained backbone gets 94% in 88
seconds and the latent path structurally cannot use one.

The costs and savings are easy to write down:

- **cost:** one conversion pass, about 0.04s per image
- **saving:** per epoch, from an input tensor 6x smaller

So it pays off only when `epochs x saving_per_epoch > conversion`, and it only
*matters* when you were going to train from scratch anyway.


In [9]:
per_image_convert = convert_seconds / 600
px_per_epoch, lt_per_epoch = px_s / 12, lt_s / 12
saved_per_epoch = px_per_epoch - lt_per_epoch

print(f'conversion       {per_image_convert*1000:.0f} ms per image, paid once')
print(f'saved per epoch  {saved_per_epoch:.1f}s on {len(datasets.ImageFolder(f"{ROOT}/train", tf))} images')
if saved_per_epoch > 0:
    print(f'break even after {convert_seconds/saved_per_epoch:.1f} epochs')
print()
for n in (600, 10_000, 100_000, 1_000_000):
    conv = n * per_image_convert
    save = n * (px_per_epoch - lt_per_epoch) / 480
    print(f'{n:>9,} images: convert {conv/60:6.1f} min, save {save/60:5.1f} min/epoch, break even after {conv/save:4.1f} epochs')


conversion       27 ms per image, paid once
saved per epoch  9.1s on 480 images
break even after 1.8 epochs

      600 images: convert    0.3 min, save   0.2 min/epoch, break even after  1.4 epochs
   10,000 images: convert    4.5 min, save   3.2 min/epoch, break even after  1.4 epochs
  100,000 images: convert   45.0 min, save  31.7 min/epoch, break even after  1.4 epochs
1,000,000 images: convert  450.4 min, save 317.4 min/epoch, break even after  1.4 epochs


## 6. The honest decision rule

**Use compressed-domain training when all of these hold:**

1. You are training **from scratch**. No pretrained backbone fits a 128-channel
   14x14 input, so if transfer learning is an option it will almost certainly win.
2. Your dataset is **large**. Tens of thousands of images upward, where the
   per-epoch saving dwarfs the one-off conversion.
3. **Disk or I/O hurts.** 6x less data is worth real money on cloud storage and
   real time on a network-mounted dataset.
4. Your task **tolerates a 16x downsample**. Classification, probably. Segmentation
   or anything needing fine spatial precision, probably not.
5. You can live **without pixel-space augmentation**. The finest crop is 16 pixels,
   colour jitter is impossible, and even a horizontal flip is not exact in latent
   space. For small datasets, losing augmentation costs more than 9x speed is worth.

**For 300 cats and 300 dogs: use a pretrained ResNet.** It takes eight lines, runs
in 88 seconds, and gets 94%.

---

*If your numbers differ from the ones at the top, trust yours. That is the point of
shipping the notebook rather than a screenshot.*
